# [F] F3: AgentTune SFT distillation of Tiny Aya Global (Kaggle GPU T4 x2)
Same steps as the kernels that `scripts/train_kaggle.py` launches: py3.12 venv, `scripts/train_sft.py` under
`torchrun` on both T4s (DDP, LoRA, assistant-only loss), checkpoints to `ToolW/desi-agent-training-logs`, resumable across Kaggle sessions.

Needs: Internet on, GPU T4 x2, and an HF token with access to Tiny Aya + write to `ToolW` (Kaggle secret `HF_TOKEN`, or a private dataset with a file `hf_token`).

In [ ]:
RUN = 'sft-a'          # same name again + RESUME=True continues the run in a new session
BRANCH = 'main'
RESUME = False
TRAIN_ARGS = ['--epochs', '2', '--push-to', 'ToolW/desi-agent-tinyaya-sft-lora', '--set', 'max_hours=11']
# F2-style smoke: TRAIN_ARGS = ['--split', 'seed', '--val-split', 'none', '--max-steps', '20']

In [ ]:
import glob, os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    os.environ['HF_TOKEN'] = open(glob.glob('/kaggle/input/**/hf_token', recursive=True)[0]).read().strip()
os.environ['DESI_HF_DATASET'] = 'ToolW/desi-bank-toolcalls'

In [ ]:
!git clone -q -b $BRANCH https://github.com/CyberRik/desi-agent.git /tmp/desi-agent
%cd /tmp/desi-agent
!git log --oneline -1

In [ ]:
!pip install -q uv
!env -u UV_SYSTEM_PYTHON -u UV_PYTHON uv venv -q -p 3.12 --python-preference managed /tmp/venv
!uv pip install --python /tmp/venv/bin/python -q --index-strategy unsafe-best-match --extra-index-url https://download.pytorch.org/whl/cu126 torch==2.11.0 -e '.[train,dev]'
!/tmp/venv/bin/python -c 'import torch, transformers, peft, agenttune; print(torch.__version__, transformers.__version__, torch.cuda.device_count())'

In [ ]:
!/tmp/venv/bin/python -m pytest -q tests/test_train_convert.py tests/test_train_chat.py tests/test_train_sft_cfg.py

In [ ]:
args = TRAIN_ARGS + ['--set', f'run_name={RUN}', '--set', 'ckpt_repo=ToolW/desi-agent-training-logs'] + (['--set', 'resume=true'] if RESUME else [])
a = ' '.join(args)
!/tmp/venv/bin/python -m torch.distributed.run --nproc_per_node 2 scripts/train_sft.py --out outputs/$RUN $a

In [ ]:
import json
m = json.load(open(f'outputs/{RUN}/manifest.json'))
print(json.dumps(m['data'], indent=1)); print(json.dumps(m['metrics'], indent=1))

In [ ]:
# quick behaviour check of the adapter on val (full eval is R's R7 via desi_agent.api.evaluate)
!/tmp/venv/bin/python scripts/train_f1_check.py --adapter outputs/$RUN/adapter --split val --limit 60 --out outputs/after_sft